# TuRBO-m 入門：複数の探索領域を競争させる

**学習目標**：複数のTrust Regionを使う理由を理解し、1領域と3領域の最良値推移をグラフで比較することです。TuRBO-1が一つの場所を重点的に調べるのに対し、TuRBO-mは複数の場所を同時に追跡します。

| 記号 | 意味 | この例 |
| --- | --- | --- |
| `m` | 探索領域の数 | 1または3 |
| `q` | 1反復で実験する合計点数 | 2 |
| `n_steps` | 反復回数 | 8 |
| `seed` | 乱数の再現設定 | 0 |
| `history` | 観測済みの最良値の推移 | グラフの縦軸 |

**比較の注意**：1領域側もTuRBO-mの管理クラスを使います。これは既存のTuRBO-1専用APIとの直接比較ではありません。

## 処理を順番に追う

1. 初期観測を用意する。
2. 領域ごとの観測データで、それぞれGPを学習する。
3. 領域ごとの中心と幅から候補範囲を決める。
4. Thompson Samplingで各領域の予測分布から候補を比較し、合計`q`点を選ぶ。
5. 選んだ候補を実際に評価する。
6. 候補を作った領域IDに従って観測を追加する。
7. 領域ごとに成功・失敗を判定し、必要なら領域幅を変えるかRestartする。

**候補の数と実験の数は違います。** 例えば`m=3,q=2`では、3領域の中から合計2点だけを実験します。2点とも同じ領域から選ばれることもあります。

このNotebookでは既存の`examples/turbo_multi_benchmark.py`を呼び出し、アルゴリズムを重複実装しません。

## 実行環境とベンチマーク

Python 3.13以降のrobotorchan環境で上から実行してください。次のセルはリポジトリ内の`examples/turbo_multi_benchmark.py`を読み込みます。ファイルが見つからなければ、リポジトリ内からNotebookを開き直してください。

比較する関数はAckley、Rosenbrock、Levyの3種類です。一般に最小化問題として扱われますが、ここでは**目的関数値の符号を反転して最大化**します。したがって**値が高いほど良い結果**です。

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from botorch.test_functions.synthetic import Ackley, Levy, Rosenbrock

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'examples' / 'turbo_multi_benchmark.py').exists()), None)
if root is None:
    raise RuntimeError('Run this notebook from the robotorchan repository.')
sys.path.insert(0, str(root / 'examples'))
from turbo_multi_benchmark import run_experiment

torch.set_default_dtype(torch.double)


## グラフの読み方

- **横軸**：追加評価の反復回数。毎回2点を評価するため、8反復では16点を追加評価します。
- **縦軸**：これまでに観測した中での最良値。高いほど良く、改善がなければ横ばいになります。
- **初期評価**：両設定とも15点。3領域では5点ずつに分けます。
- **線の違い**：1領域と3領域を比較します。領域数が多ければ必ず優れるわけではありません。

**結果を読むときの注意**：seedが一つだけなので、グラフ1本から一般的な優劣は判断できません。複数seedで繰り返し、最終最良値の平均・ばらつきや改善の速さを比較してください。

In [ ]:
for problem in (Ackley(dim=2), Rosenbrock(dim=2), Levy(dim=2)):
    fig, ax = plt.subplots(figsize=(6, 3.5))
    for n_regions in (1, 3):
        history = run_experiment(
            problem, n_regions=n_regions, q=2, n_steps=8,
            initial_per_region=15 // n_regions, seed=0,
        )
        ax.plot(range(len(history)), history, marker='o', label=f'{n_regions} region(s)')
    ax.set(title=type(problem).__name__, xlabel='BO iteration', ylabel='Best negative objective')
    ax.legend()
    plt.show()


## 理解度チェック

**Q1.** `m=3,q=2`で3領域すべてが必ず評価されますか？ → **いいえ**。評価されない領域もあります。

**Q2.** グラフの縦軸が高い方が良いのはなぜですか？ → **最小化関数の符号を反転して最大化しているから**です。

**Q3.** 3領域の方が1回だけ良かったら、TuRBO-mの方が優れていると結論できますか？ → **いいえ**。複数seedの検証が必要です。

続けて[TuRBO-m理論](../../docs/theory/optimization/10_trust_region.md)と[TuRBO-1のNotebook](34_turbo_trust_region.ipynb)を参照してください。